In [1]:
from pyspark.sql.functions import (
    col, trim, lower, upper, initcap, to_timestamp, 
    when, round as spark_round, current_timestamp
)
from pyspark.sql.types import DecimalType

# حدود جغرافية تقريبية للبرازيل (لفلترة الإحداثيات الغلط)
LAT_MIN, LAT_MAX = -34.0, 6.0
LNG_MIN, LNG_MAX = -74.0, -32.0

StatementMeta(, e0d9a74d-c1d0-4af2-a5fc-45ba0cc44c43, 3, Finished, Available, Finished, False)

### Customers Cleanning

In [2]:
df_customers = spark.table("ods.ods_customers")

stg_customers = (df_customers
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_unique_id", trim(col("customer_unique_id")))
    .withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("string"))
    .withColumn("customer_city", initcap(trim(col("customer_city"))))
    .withColumn("customer_state", upper(trim(col("customer_state"))))
    .dropDuplicates(["customer_id"])
    .withColumn("_stg_load_timestamp", current_timestamp())
    .select("customer_id", "customer_unique_id", "customer_zip_code_prefix",
            "customer_city", "customer_state", "_stg_load_timestamp")
)

stg_customers.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("stg.stg_customers")
print(f" stg_customers: {stg_customers.count()} rows")

StatementMeta(, e0d9a74d-c1d0-4af2-a5fc-45ba0cc44c43, 4, Finished, Available, Finished, False)

 stg_customers: 99441 rows


### Geolocation Cleanning

In [3]:
df_geo = spark.table("ods.ods_geolocation")

stg_geolocation = (df_geo
    .withColumn("geolocation_zip_code_prefix", col("geolocation_zip_code_prefix").cast("string"))
    .withColumn("geolocation_city", initcap(trim(col("geolocation_city"))))
    .withColumn("geolocation_state", upper(trim(col("geolocation_state"))))
    # استبعاد الإحداثيات الغلط (خارج حدود البرازيل)
    .filter(
        (col("geolocation_lat").between(LAT_MIN, LAT_MAX)) &
        (col("geolocation_lng").between(LNG_MIN, LNG_MAX))
    )
    .dropDuplicates()  # فيه تكرار كامل كتير في المصدر
    .withColumn("_stg_load_timestamp", current_timestamp())
)

stg_geolocation.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("stg.stg_geolocation")
print(f" stg_geolocation: {stg_geolocation.count()} rows")

StatementMeta(, e0d9a74d-c1d0-4af2-a5fc-45ba0cc44c43, 5, Finished, Available, Finished, False)

 stg_geolocation: 738305 rows


### Sellers Cleanning

In [4]:
df_sellers = spark.table("ods.ods_sellers")

stg_sellers = (df_sellers
    .withColumn("seller_id", trim(col("seller_id")))
    .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("string"))
    .withColumn("seller_city", initcap(trim(col("seller_city"))))
    .withColumn("seller_state", upper(trim(col("seller_state"))))
    .dropDuplicates(["seller_id"])
    .withColumn("_stg_load_timestamp", current_timestamp())
)

stg_sellers.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("stg.stg_sellers")
print(f" stg_sellers: {stg_sellers.count()} rows")

StatementMeta(, e0d9a74d-c1d0-4af2-a5fc-45ba0cc44c43, 6, Finished, Available, Finished, False)

 stg_sellers: 3095 rows


### Product_Category_Name_Translation + Products ....... Cleanning

In [5]:
df_translation = spark.table("ods.ods_product_category_translation")
df_products = spark.table("ods.ods_products")

stg_translation = (df_translation
    .withColumn("product_category_name", trim(col("product_category_name")))
    .withColumn("product_category_name_english", trim(col("product_category_name_english")))
    .dropDuplicates(["product_category_name"])
)
stg_translation.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("stg.stg_product_category_translation")

stg_products = (df_products
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_category_name", trim(col("product_category_name")))
    .join(stg_translation, on="product_category_name", how="left")
    # معالجة الفئات الناقصة أو غير المترجمة
    .withColumn("product_category_name_english",
                when(col("product_category_name_english").isNull(), "unknown")
                .otherwise(col("product_category_name_english")))
    .withColumn("product_name_lenght", col("product_name_lenght").cast("int"))
    .withColumn("product_description_lenght", col("product_description_lenght").cast("int"))
    .withColumn("product_photos_qty", col("product_photos_qty").cast("int"))
    .withColumn("product_weight_g", col("product_weight_g").cast("double"))
    .withColumn("product_length_cm", col("product_length_cm").cast("double"))
    .withColumn("product_height_cm", col("product_height_cm").cast("double"))
    .withColumn("product_width_cm", col("product_width_cm").cast("double"))
    .dropDuplicates(["product_id"])
    .withColumn("_stg_load_timestamp", current_timestamp())
    .select("product_id", "product_category_name_english", "product_name_lenght",
            "product_description_lenght", "product_photos_qty", "product_weight_g",
            "product_length_cm", "product_height_cm", "product_width_cm", "_stg_load_timestamp")
)

stg_products.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("stg.stg_products")
print(f" stg_products: {stg_products.count()} rows")

StatementMeta(, e0d9a74d-c1d0-4af2-a5fc-45ba0cc44c43, 7, Finished, Available, Finished, False)

 stg_products: 32951 rows


### Orders Cleanning

In [6]:
df_orders = spark.table("ods.ods_orders")

stg_orders = (df_orders
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("order_status", lower(trim(col("order_status"))))
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp")))
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at")))
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date")))
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date")))
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date")))
    .dropDuplicates(["order_id"])
    .withColumn("_stg_load_timestamp", current_timestamp())
)

stg_orders.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("stg.stg_orders")
print(f" stg_orders: {stg_orders.count()} rows")

StatementMeta(, e0d9a74d-c1d0-4af2-a5fc-45ba0cc44c43, 8, Finished, Available, Finished, False)

 stg_orders: 99441 rows


### Order_items Cleanning

In [7]:
df_items = spark.table("ods.ods_order_items")

stg_order_items = (df_items
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("order_item_id", col("order_item_id").cast("int"))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("seller_id", trim(col("seller_id")))
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date")))
    .withColumn("price", spark_round(col("price").cast(DecimalType(10,2)), 2))
    .withColumn("freight_value", spark_round(col("freight_value").cast(DecimalType(10,2)), 2))
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("_stg_load_timestamp", current_timestamp())
)

stg_order_items.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("stg.stg_order_items")
print(f" stg_order_items: {stg_order_items.count()} rows")

StatementMeta(, e0d9a74d-c1d0-4af2-a5fc-45ba0cc44c43, 9, Finished, Available, Finished, False)

 stg_order_items: 112650 rows


### Order_Payments Cleanning

In [8]:
df_payments = spark.table("ods.ods_order_payments")

stg_order_payments = (df_payments
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("payment_sequential", col("payment_sequential").cast("int"))
    .withColumn("payment_type", lower(trim(col("payment_type"))))
    .withColumn("payment_installments", col("payment_installments").cast("int"))
    .withColumn("payment_value", spark_round(col("payment_value").cast(DecimalType(10,2)), 2))
    .dropDuplicates(["order_id", "payment_sequential"])
    .withColumn("_stg_load_timestamp", current_timestamp())
)

stg_order_payments.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("stg.stg_order_payments")
print(f" stg_order_payments: {stg_order_payments.count()} rows")

StatementMeta(, e0d9a74d-c1d0-4af2-a5fc-45ba0cc44c43, 10, Finished, Available, Finished, False)

 stg_order_payments: 103886 rows


### Order_Reviews Cleanning

In [9]:
df_reviews = spark.table("ods.ods_order_reviews")

stg_order_reviews = (df_reviews
    .withColumn("review_id", trim(col("review_id")))
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("review_score", col("review_score").cast("int"))
    .withColumn("review_comment_title", trim(col("review_comment_title")))
    .withColumn("review_comment_message", trim(col("review_comment_message")))
    .withColumn("review_creation_date", to_timestamp(col("review_creation_date")))
    .withColumn("review_answer_timestamp", to_timestamp(col("review_answer_timestamp")))
    .dropDuplicates(["review_id"])
    .withColumn("_stg_load_timestamp", current_timestamp())
)

stg_order_reviews.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("stg.stg_order_reviews")
print(f" stg_order_reviews: {stg_order_reviews.count()} rows")

StatementMeta(, e0d9a74d-c1d0-4af2-a5fc-45ba0cc44c43, 11, Finished, Available, Finished, False)

 stg_order_reviews: 98410 rows
